In [ ]:
# NST FEVER Training — Local GPU Execution

**Neurosymbolic Transformers for FEVER Fact Verification**

This notebook runs the full training pipeline on local hardware (Apple Silicon MPS or CUDA GPU).

## Execution plan
1. **Cell 1**: Environment setup & GPU detection
2. **Cell 2**: Data loading & sanity check
3. **Cell 3**: Smoke test (200 examples, validates full pipeline)
4. **Cell 4**: Neural baseline (DeBERTa-v3-base, gold evidence)
5. **Cell 5**: NST-VERI flagship (3-phase verification-enhanced training)
6. **Cell 6**: Results analysis & comparison
7. **Cell 7**: Save results

**Honesty policy**: Only real measured numbers. No fake metrics.

Wed Apr 15 22:09:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
# ============================================================
# Cell 1: Environment Setup & GPU Detection
# ============================================================
import os, sys, time, gc, pathlib

# ── Find project root ──
# Strategy: look for data/ and models/ directories to identify root.
# Works in Colab (/content/nst), local dev, or any layout.
_candidates = [
    os.getcwd(),
    os.path.dirname(os.path.abspath("__file__")),  # notebook dir (local)
    "/content/nst",       # Colab default after git clone
    "/content/Neurosymbolic-Transformers",
    "/content",
]
PROJ_ROOT = None
for c in _candidates:
    if os.path.isdir(os.path.join(c, "data")) and os.path.isdir(os.path.join(c, "models")):
        PROJ_ROOT = c
        break

if PROJ_ROOT is None:
    # If repo isn't cloned yet (Colab), clone it
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git",
                    "/content/nst"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/nst"], check=True)
    PROJ_ROOT = "/content/nst"

if PROJ_ROOT not in sys.path:
    sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)
print(f"Project root: {PROJ_ROOT}")

# Verify core imports
import torch
import transformers
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")

# ═══════════════════════════════════════════════════════════
#  GPU Auto-Detection
# ═══════════════════════════════════════════════════════════
GPU_OVERRIDES = {}

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    cc = (props.major, props.minor)
    supports_bf16 = cc >= (8, 0)

    if vram_gb >= 35:
        BS, GA = 32, 2
    elif vram_gb >= 20:
        BS, GA = 24, 2
    else:
        BS, GA = 16, 2

    GPU_OVERRIDES = {
        "train": {
            "batch_size": BS,
            "grad_accum_steps": GA,
            "bf16": supports_bf16,
            "fp16": not supports_bf16,
            "tf32": supports_bf16,
            "fused_optimizer": True,
            "num_workers": 4,
        }
    }
    if supports_bf16:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    DEVICE = "cuda"
    DEVICE_NAME = f"{gpu_name} ({vram_gb:.0f}GB)"
    prec = "BF16" if supports_bf16 else "FP16"

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    # Apple Silicon — shared memory, smaller batches
    import subprocess
    r = subprocess.run(["sysctl", "-n", "hw.memsize"], capture_output=True, text=True)
    ram_gb = int(r.stdout.strip()) / 1e9
    r2 = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
    chip = r2.stdout.strip()

    if ram_gb >= 32:
        BS, GA = 16, 2
    elif ram_gb >= 16:
        BS, GA = 8, 4
    else:
        BS, GA = 4, 8

    GPU_OVERRIDES = {
        "train": {
            "batch_size": BS,
            "grad_accum_steps": GA,
            "bf16": False,
            "fp16": True,
            "num_workers": 0,
        }
    }

    DEVICE = "mps"
    DEVICE_NAME = f"{chip} ({ram_gb:.0f}GB shared)"
    prec = "FP16"
else:
    DEVICE = "cpu"
    DEVICE_NAME = "CPU (no GPU)"
    BS, GA = 4, 8
    prec = "FP32"
    GPU_OVERRIDES = {
        "train": {
            "batch_size": BS,
            "grad_accum_steps": GA,
            "bf16": False,
            "fp16": False,
            "num_workers": 0,
        }
    }

eff_bs = BS * GA
print(f"\n{'='*60}")
print(f"  Device    : {DEVICE_NAME}")
print(f"  Backend   : {DEVICE}")
print(f"  Batch     : {BS} x {GA} = {eff_bs} effective")
print(f"  Precision : {prec}")
print(f"{'='*60}")

Project root: /content/nst
PyTorch      : 2.9.0+cu126
Transformers : 4.57.6

  Device    : NVIDIA A100-SXM4-40GB (42GB)
  Backend   : cuda
  Batch     : 32 x 2 = 64 effective
  Precision : BF16


/usr/local/lib/python3.12/dist-packages/torch/backends/__init__.py:46: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  self.setter(val)


In [2]:
# ============================================================
# Cell 2: Verify datasets version is compatible
# ============================================================
# datasets 2.21.0 should already be installed (was installed above
# with kernel restart). Verify it.
import datasets
print(f"datasets version: {datasets.__version__}")
_ds_major = int(datasets.__version__.split(".")[0])
assert _ds_major < 3, (
    f"datasets {datasets.__version__} doesn't support FEVER loading scripts. "
    "Run: pip install datasets==2.21.0 then restart the kernel."
)
print("OK — compatible with FEVER loading script")

datasets version: 2.21.0
OK — compatible with FEVER loading script


In [4]:
# ============================================================
# Cell 3: Build Wiki Cache & Load Data
# ============================================================
# The FEVER dataset needs a wiki page cache to resolve evidence
# text from page title + sentence index. Without it, we only
# get page titles as "evidence", which cripples NLI training.
import logging, os, time
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Clear stale project modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

# ── Step 1: Build wiki cache if missing ──
from data.fever_wiki_cache import build_wiki_cache, cache_stats

cache_path = os.path.join(PROJ_ROOT, "data", "fever_wiki.db")
stats = cache_stats(cache_path)
if stats.get("exists"):
    print(f"Wiki cache exists: {stats['n_pages']} pages, {stats['size_mb']:.1f} MB")
else:
    print("Building wiki page cache (one-time, ~5-10 min on Colab)...")
    print("  This downloads ~1.7GB of Wikipedia pages and indexes ~25K needed pages.\n")
    t0 = time.time()
    build_stats = build_wiki_cache(cache_path=cache_path)
    elapsed = time.time() - t0
    print(f"\n  Done in {elapsed:.0f}s: {build_stats['n_found']}/{build_stats['n_needed']} pages "
          f"({build_stats['n_missing']} missing)")
    print(f"  Cache: {cache_path} ({build_stats['cache_size_mb']:.1f} MB)")

# ── Step 2: Load data with evidence text ──
from data.fever_dataset import load_fever_splits, print_fever_stats

splits_check = load_fever_splits(max_train=500, max_dev=200, dev_test_ratio=0.1, seed=42)
print_fever_stats(splits_check)

train_items = splits_check["train"]
if len(train_items) == 0:
    print("\n  ERROR: No training data loaded!")
else:
    n_with_evidence = sum(1 for it in train_items if len(it.get("gold_evidence_text", "")) > 30)
    pct = 100 * n_with_evidence / len(train_items)
    print(f"\n  Evidence quality: {n_with_evidence}/{len(train_items)} ({pct:.0f}%) have >30 char evidence")

    for i in [0, 1, 2]:
        it = train_items[i]
        ev = it.get("gold_evidence_text", it.get("evidence", ""))[:150]
        print(f"\n  [{i}] {it['claim'][:80]}")
        print(f"      Label: {it['label']}")
        print(f"      Evidence: {ev}")

    if pct > 60:
        print(f"\n  Data check PASSED — good evidence coverage")
    elif pct > 20:
        print(f"\n  Data check WARNING — partial evidence ({pct:.0f}%)")
    else:
        print(f"\n  Data check FAILED — only {pct:.0f}% have evidence. Wiki cache needed.")

fever_wiki_cache | Building FEVER wiki cache...
fever_wiki_cache |   Loading HF fever/v1.0 dataset...


Building wiki page cache (one-time, ~5-10 min on Colab)...
  This downloads ~1.7GB of Wikipedia pages and indexes ~25K needed pages.



fever_wiki_cache |   Pass 1: Scanning annotations for needed wiki page titles...
fever_wiki_cache |   Found 14533 unique page titles in annotations
fever_wiki_cache |   wiki_pages not in v1.0 — loading fever/wiki_pages separately...


Generating wikipedia_pages split:   0%|          | 0/5416537 [00:00<?, ? examples/s]

fever_wiki_cache |   Pass 2: Streaming 5416537 wiki pages, filtering to 14533 needed titles...
fever_wiki_cache |     Scanned 1894767 pages, found 5000/14533...
fever_wiki_cache |     Scanned 3951899 pages, found 10000/14533...
fever_wiki_cache |   ✅ Wiki cache built: 14363/14533 pages (170 missing) in 406.8s → /content/nst/data/fever_wiki.db (24.2 MB)
fever_wiki_cache |   ⚠️  170 pages not found in wiki_pages split. Evidence for those will use title-only fallback.
fever_wiki_cache |   Manifest written to /content/nst/data/fever_wiki_manifest.json
fever_dataset | Loading FEVER from HuggingFace datasets...



  Done in 407s: 14363/14533 pages (170 missing)
  Cache: /content/nst/data/fever_wiki.db (24.2 MB)


fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 500 examples (291 with evidence text, 209 without)
fever_dataset |   dev: 200 examples (87 with evidence text, 113 without)
fever_dataset |   Split labelled_dev into dev (180) + dev_test (20)
fever_dataset |   train hash: 1a8f068b90ef1d45
fever_dataset |   dev hash: de31c0c87be71ed3
fever_dataset |   dev_test hash: 46ca4d6615bf52ec


  FEVER Dataset Statistics

  train: 500 examples
    With gold evidence: 291 (58.2%)
    Label distribution:
      SUPPORTS                351  (70.2%)
      REFUTES                  50  (10.0%)
      NOT ENOUGH INFO          99  (19.8%)
    Split hash: 1a8f068b90ef1d45

  dev: 180 examples
    With gold evidence: 80 (44.4%)
    Label distribution:
      SUPPORTS                 95  (52.8%)
      REFUTES                  37  (20.6%)
      NOT ENOUGH INFO          48  (26.7%)
    Split hash: de31c0c87be71ed3

  dev_test: 20 examples
    With gold evidence: 7 (35.0%)
    Label distribution:
      SUPPORTS                  7  (35.0%)
      REFUTES                   6  (30.0%)
      NOT ENOUGH INFO           7  (35.0%)
    Split hash: 46ca4d6615bf52ec

  Evidence quality: 289/500 (58%) have >30 char evidence

  [0] Chris Hemsworth appeared in A Perfect Getaway.
      Label: SUPPORTS
      Evidence: Hemsworth has also appeared in the science fiction action film Star Trek -LRB- 2009 -RRB- ,

In [5]:
# ============================================================
# Cell 4: Smoke Test — 200 examples (validates pipeline end-to-end)
# ============================================================
import time, gc, json

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("=" * 60)
print("  SMOKE TEST: 200 train / 100 dev / 1 epoch")
print(f"  Device: {DEVICE_NAME}")
print("=" * 60 + "\n")

# Use same precision as real training to validate that path
_use_bf16 = GPU_OVERRIDES.get("train", {}).get("bf16", False)
_use_fp16 = GPU_OVERRIDES.get("train", {}).get("fp16", False)

SMOKE_OVERRIDES = {
    "data": {"max_train": 200, "max_dev": 100},
    "train": {
        "epochs": 1,
        "batch_size": min(BS, 16),
        "grad_accum_steps": 1,
        "eval_every_steps": 50,
        "patience": 99,
        "bf16": _use_bf16,
        "fp16": _use_fp16 and not _use_bf16,
        "num_workers": 0 if DEVICE == "mps" else 2,
    },
    "model": {"max_length": 128, "gradient_checkpointing": False},
    "io": {"out_dir": "outputs_smoke"},
}

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_smoke = train_fever_nst("configs/fever_gold_neural.yaml",
                                 config_overrides=SMOKE_OVERRIDES)
elapsed = time.time() - t0

dev = results_smoke.get("dev", {})
print(f"\n{'='*60}")
print(f"  SMOKE TEST RESULTS ({elapsed:.0f}s)")
print(f"{'='*60}")
print(f"  dev_accuracy  : {dev.get('accuracy', 'N/A')}")
print(f"  nan_abort     : {results_smoke.get('nan_abort', False)}")
print(f"  trainable     : {results_smoke.get('trainable_params_M', '?')}M / {results_smoke.get('total_params_M', '?')}M")

if results_smoke.get("nan_abort", False):
    print("\n  SMOKE TEST FAILED — NaN detected!")
else:
    print("\n  Smoke test PASSED — pipeline is working")

with open("results_smoke.json", "w") as f:
    json.dump(results_smoke, f, indent=2, default=str)

gc.collect()

  SMOKE TEST: 200 train / 100 dev / 1 epoch
  Device: NVIDIA A100-SXM4-40GB (42GB)



train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...
fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 200 examples (105 with evidence text, 95 without)
fever_dataset |   dev: 100 examples (47 with evidence text, 53 without)
fever_dataset |   Split labelled_dev into dev (90) + dev_test (10)
fever_dataset |   train hash: 398028c26c5ec3e7
fever_dataset |   dev hash: 6499c2440bedf588
fever_dataset |   dev_test hash: 8e0f54c7ad68ac9e
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 200 examples
    With gold evidence: 105 (52.5%)
    Label distribution:
      SUPPORTS                129  (64.5%)
      REFUTES                  16  (8.0%)
      NOT ENOUGH INFO          55  (27.5%)
    Split hash: 398028c26c5ec3e7

  dev: 90 examples
    With gold evidence: 41 (45.6%)
    Label distribution:
      SUPPORTS                 41  (45.6%)
      REFUTES                  28  (31.1%)
      NOT ENOUGH INFO          21  (23.3%)
    Split hash: 6499c2440bedf588

  dev_test: 10 examples
    With gold evidence: 6 (60.0%)
    Label distribution:
      SUPPORTS                  5  (50.0%)
      REFUTES                   2  (20.0%)
      NOT ENOUGH INFO           3  (30.0%)
    Split hash: 8e0f54c7ad68ac9e


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Epoch 1/1: loss=1.1543 constraint=0.0000 | dev_acc=0.3111 ECE=0.0626

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────
  Learned temperature: T = 2.3038

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────
  Label Accuracy (GOLD evidence): 0.3111
  ECE: 0.0626
  Brier: 0.6864
    SUPPORTS: acc=0.0000 (n=41)
    REFUTES: acc=1.0000 (n=28)
    NOT ENOUGH INFO: acc=0.0000 (n=21)

────────────────────────────────────────
  Final evaluation on held-out dev_test
────────────────────────────────────────


train_fever | Report saved to outputs_smoke/report.json


  Label Accuracy (GOLD evidence): 0.2000
  ECE: 0.1734
  Brier: 0.6970
    SUPPORTS: acc=0.0000 (n=5)
    REFUTES: acc=1.0000 (n=2)
    NOT ENOUGH INFO: acc=0.0000 (n=3)

  Training complete in 6.6s
  Best dev accuracy: 0.3111
  Output: outputs_smoke

  SMOKE TEST RESULTS (60s)
  dev_accuracy  : 0.3111
  nan_abort     : False
  trainable     : 184.42M / 184.42M

  Smoke test PASSED — pipeline is working


291

In [6]:
# ============================================================
# Cell 4: Neural Baseline — DeBERTa-v3-base (gold evidence)
# ============================================================
# Pure neural NLI baseline. No symbolic constraints.
# DeBERTa-v3-base on full (or subset) FEVER.
# This establishes the baseline accuracy to beat.
import time, json, gc

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Choose dataset size based on device ──
# MPS/16GB: use 10K subset for reasonable training time (~20-30 min)
# CUDA: use full dataset
if DEVICE == "mps":
    MAX_TRAIN = 10000
    MAX_DEV = 2000
    EPOCHS = 3
    EVAL_EVERY = 200
    est_time = "~20-30 min"
elif DEVICE == "cuda":
    MAX_TRAIN = None  # full ~145K
    MAX_DEV = None
    EPOCHS = 3
    EVAL_EVERY = 500
    est_time = "~25-40 min"
else:
    MAX_TRAIN = 2000
    MAX_DEV = 500
    EPOCHS = 2
    EVAL_EVERY = 100
    est_time = "~30-60 min"

BASELINE_OVERRIDES = {
    "data": {"max_train": MAX_TRAIN, "max_dev": MAX_DEV},
    "train": {
        "epochs": EPOCHS,
        "eval_every_steps": EVAL_EVERY,
        **GPU_OVERRIDES.get("train", {}),
    },
    "io": {"out_dir": "outputs_fever_neural_baseline"},
}
# Ensure correct model for memory-constrained devices
if DEVICE in ("mps", "cpu"):
    BASELINE_OVERRIDES["model"] = {
        "name": "microsoft/deberta-v3-base",
        "use_lora": False,
        "gradient_checkpointing": True,
        "max_length": 384,
    }

n_train_str = f"{MAX_TRAIN//1000}K" if MAX_TRAIN else "full (~145K)"
print("=" * 65)
print("  NEURAL BASELINE: DeBERTa-v3-base")
print("  Gold Evidence, Setting A")
print("=" * 65)
print(f"  Device    : {DEVICE_NAME}")
print(f"  Train set : {n_train_str}")
print(f"  Epochs    : {EPOCHS}")
print(f"  Batch     : {BASELINE_OVERRIDES['train'].get('batch_size', BS)} x "
      f"{BASELINE_OVERRIDES['train'].get('grad_accum_steps', GA)}")
print(f"  Est time  : {est_time}")
print("=" * 65 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_baseline = train_fever_nst("configs/fever_gold_neural.yaml",
                                    config_overrides=BASELINE_OVERRIDES)
elapsed_baseline = time.time() - t0

dev = results_baseline.get("dev", {})
dt = results_baseline.get("dev_test", {})
print(f"\n{'='*65}")
print(f"  NEURAL BASELINE RESULTS ({elapsed_baseline/60:.1f} min)")
print(f"{'='*65}")
print(f"  Train size : {n_train_str}")
print(f"  Dev acc    : {dev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {dev.get('ece', 'N/A')}")
if dt:
    print(f"  DevTest acc: {dt.get('accuracy', 'N/A')}  (held-out)")
print(f"  Best dev   : {results_baseline.get('best_dev_acc', 'N/A')}")
print(f"  Temperature: {results_baseline.get('temperature', 'N/A')}")
print(f"\n  Per-label (dev):")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_baseline.json", "w") as f:
    json.dump(results_baseline, f, indent=2, default=str)
print(f"\n  Saved to results_baseline.json")

gc.collect()

train_fever | TF32 enabled for matmul and cuDNN
train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NEURAL BASELINE: DeBERTa-v3-base
  Gold Evidence, Setting A
  Device    : NVIDIA A100-SXM4-40GB (42GB)
  Train set : full (~145K)
  Epochs    : 3
  Batch     : 32 x 2
  Est time  : ~25-40 min



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M total, 184.4M trainable)
train_fever | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever | Periodic eval uses dev subset: 2000/17998 examples
train_fever | Params: 184.42M trainable / 184.42M total
train_fever |   backbone: 184.42M params, lr=2e-05
train_fever | Using fused AdamW



  FEVER Training: mode=neural
  Model: microsoft/deberta-v3-base (full FT)
  Trainable: 184.42M / 184.42M (100.0%)
  epochs=3, bs=32x2=64, lr=2e-05
  evidence_mode=gold, precision=bf16, compile=False
  total_steps=6819, warmup=409

  Step 500: loss=0.5712 | dev_acc=0.7975 ECE=0.0200
  Step 1000: loss=0.6587 | dev_acc=0.8150 ECE=0.0217
  Step 1500: loss=0.4131 | dev_acc=0.8310 ECE=0.0299
  Step 2000: loss=0.5390 | dev_acc=0.8345 ECE=0.0296
  Epoch 1/3: loss=0.5513 constraint=0.0000
  Step 2500: loss=0.5321 | dev_acc=0.8345 ECE=0.0286
  Step 3000: loss=0.4527 | dev_acc=0.8285 ECE=0.0428
  Step 3500: loss=0.3536 | dev_acc=0.8380 ECE=0.0345
  Step 4000: loss=0.4942 | dev_acc=0.8345 ECE=0.0414
  Step 4500: loss=0.3458 | dev_acc=0.8345 ECE=0.0318
  Epoch 2/3: loss=0.4246 constraint=0.0000
  Step 5000: loss=0.2889 | dev_acc=0.8390 ECE=0.0430
  Step 5500: loss=0.3207 | dev_acc=0.8420 ECE=0.0464
  Step 6000: loss=0.2190 | dev_acc=0.8385 ECE=0.0398
  Step 6500: loss=0.2953 | dev_acc=0.8435 ECE=

train_fever | Report saved to outputs_fever_neural_baseline/report.json


  Label Accuracy (GOLD evidence): 0.8350
  ECE: 0.0398
  Brier: 0.2430
    SUPPORTS: acc=0.8528 (n=652)
    REFUTES: acc=0.8336 (n=697)
    NOT ENOUGH INFO: acc=0.8187 (n=651)

  Training complete in 2226.4s
  Best dev accuracy: 0.8435
  Output: outputs_fever_neural_baseline

  NEURAL BASELINE RESULTS (38.5 min)
  Train size : full (~145K)
  Dev acc    : 0.8378
  Dev ECE    : 0.040138
  DevTest acc: 0.835  (held-out)
  Best dev   : 0.8435
  Temperature: 1.2331

  Per-label (dev):
    SUPPORTS            : 0.8670 (n=6014)
    REFUTES             : 0.8202 (n=5969)
    NOT ENOUGH INFO     : 0.8259 (n=6015)

  Saved to results_baseline.json


111

In [7]:
# ============================================================
# Cell 6: NST-VERI — Neurosymbolic DeBERTa + CEGIS verification
# ============================================================
# Full neurosymbolic pipeline: DeBERTa backbone + constraint head
# + CEGIS counterexample loop. This is the novel contribution.
# Uses DeBERTa-v3-large + LoRA (vs base for baseline).
import time, json, gc

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# NST-VERI uses its own epoch count (5 = 3 phases) and eval settings.
# We only override GPU hardware settings and data limits.
VERI_OVERRIDES = {
    "data": {"max_train": MAX_TRAIN, "max_dev": MAX_DEV},
    "train": {
        # Use config's epochs (5) — needed for 3-phase training
        **{k: v for k, v in GPU_OVERRIDES.get("train", {}).items()
           if k not in ("epochs",)},  # preserve config epochs
    },
    "io": {"out_dir": "outputs_fever_nst_veri"},
}
if DEVICE in ("mps", "cpu"):
    VERI_OVERRIDES["model"] = {
        "name": "microsoft/deberta-v3-base",
        "use_lora": False,
        "gradient_checkpointing": True,
        "max_length": 384,
    }

print("=" * 65)
print("  NST-VERI: DeBERTa-v3-large + LoRA + CEGIS Verification")
print("  Gold Evidence, Setting A")
print("=" * 65)
print(f"  Device    : {DEVICE_NAME}")
print(f"  Train set : {n_train_str}")
print(f"  Epochs    : 5 (3-phase: NLI→contrastive→constraints)")
print(f"  Model     : DeBERTa-v3-large + LoRA (rank=16)")
print("=" * 65 + "\n")

t0 = time.time()
from training.train_fever_veri import train_fever_veri
results_veri = train_fever_veri("configs/fever_gold_nst_veri.yaml",
                                 config_overrides=VERI_OVERRIDES)
elapsed_veri = time.time() - t0

vdev = results_veri.get("dev", {})
vdt = results_veri.get("dev_test", {})
print(f"\n{'='*65}")
print(f"  NST-VERI RESULTS ({elapsed_veri/60:.1f} min)")
print(f"{'='*65}")
print(f"  Train size : {n_train_str}")
print(f"  Dev acc    : {vdev.get('accuracy', 'N/A')}")
print(f"  Dev ECE    : {vdev.get('ece', 'N/A')}")
if vdt:
    print(f"  DevTest acc: {vdt.get('accuracy', 'N/A')}  (held-out)")
print(f"  Best dev   : {results_veri.get('best_dev_acc', 'N/A')}")
print(f"  Temperature: {results_veri.get('temperature', 'N/A')}")
viol = results_veri.get("constraint_violations", {})
if viol:
    print(f"  Violations : {viol}")
print(f"\n  Per-label (dev):")
for label, stats in vdev.get("per_label", {}).items():
    print(f"    {label:<20}: {stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_veri.json", "w") as f:
    json.dump(results_veri, f, indent=2, default=str)
print(f"\n  Saved to results_veri.json")

gc.collect()

train_fever_veri | TF32 enabled
train_fever_veri | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NST-VERI: DeBERTa-v3-large + LoRA + CEGIS Verification
  Gold Evidence, Setting A
  Device    : NVIDIA A100-SXM4-40GB (42GB)
  Train set : full (~145K)
  Epochs    : 5 (3-phase: NLI→contrastive→constraints)
  Model     : DeBERTa-v3-large + LoRA (rank=16)



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever_veri | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-large


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

fever_nli | LoRA applied: r=16, alpha=32, trainable=7.1M / 442.2M (1.6%)
fever_nli | Model loaded: microsoft/deberta-v3-large (442.2M total, 7.1M trainable)
train_fever_veri | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever_veri |   backbone: 0.00M params, lr=1e-05
train_fever_veri |   lora: 7.11M params, lr=0.0003
train_fever_veri |   heads: 2.43M params, lr=0.0005
train_fever_veri |   adaptive_lambda: 0.00M params, lr=0.001



  NST-VERI Training: Verification-Enhanced FEVER
  Model: microsoft/deberta-v3-large + LoRA r=16
  Trainable: 9.55M / 444.61M
  Adaptive lambda: 4551 params, lambda_max=0.3
  epochs=5, bs=32x2=64
  lr=1e-05, lr_lora=0.0003, lr_heads=0.0005
  precision=bf16, compile=False
  Focal loss: no
  Phases: 1→NLI+aux, 2→+contrastive, 3→+constraints
  total_steps=11365, warmup=681

  Phase 1 | Epoch 1/5 | β=1.00 γ=0.00 sched=0.00
    Step 250: loss=1.2642 nli=0.6437 aux=0.6205 con=0.0000 cst=0.0000 | dev_acc=0.7775 ECE=0.0351 res_scale=0.0481
    Step 500: loss=1.2875 nli=0.6664 aux=0.6211 con=0.0000 cst=0.0000 | dev_acc=0.8125 ECE=0.0337 res_scale=0.0481
    Step 750: loss=1.1044 nli=0.5129 aux=0.5915 con=0.0000 cst=0.0000 | dev_acc=0.8260 ECE=0.0272 res_scale=0.0478
    Step 1000: loss=1.2078 nli=0.6048 aux=0.6030 con=0.0000 cst=0.0000 | dev_acc=0.8265 ECE=0.0210 res_scale=0.0477
    Step 1250: loss=0.9656 nli=0.3587 aux=0.6068 con=0.0000 cst=0.0000 | dev_acc=0.8325 ECE=0.0202 res_scale=0.0478

train_fever_veri | Early stopping at step 5000


    Step 5000: loss=0.9150 nli=0.3026 aux=0.5977 con=0.1467 cst=0.0000 | dev_acc=0.8400 ECE=0.0471 res_scale=0.0476

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────
  Learned temperature: T = 1.1949

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────
  Label Accuracy (GOLD evidence): 0.8384
  ECE: 0.0423
  Brier: 0.2369
    SUPPORTS: acc=0.8964 (n=6014)
    REFUTES: acc=0.7953 (n=5969)
    NOT ENOUGH INFO: acc=0.8231 (n=6015)

────────────────────────────────────────
  Final evaluation on held-out dev_test
────────────────────────────────────────
  Label Accuracy: 0.8320
  ECE: 0.0457
    SUPPORTS: acc=0.8850 (n=652)
    REFUTES: acc=0.8149 (n=697)
    NOT ENOUGH INFO: acc=0.7972 (n=651)

  Training complete in 7234.0s
  Best dev accuracy: 0.8525
  Output: outputs_fever_nst_veri

  NST-VERI RESULTS (124.3 min)
  Train size : full (~145K)
  Dev acc    :

111

In [8]:
# ============================================================
# Cell 6: Honest Results Comparison
# ============================================================
import json, os
from datetime import datetime

# ── Gather all results ──
all_results = {}
for name, path in [("smoke", "results_smoke.json"),
                    ("neural_baseline", "results_baseline.json"),
                    ("nst_veri", "results_veri.json")]:
    if os.path.exists(path):
        with open(path) as f:
            all_results[name] = json.load(f)

print("=" * 72)
print("  HONEST RESULTS COMPARISON")
print(f"  Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Device: {DEVICE_NAME}")
print("=" * 72)

# ── Results table ──
header = f"{'Model':<25} {'Dev Acc':>10} {'Dev ECE':>10} {'DevTest':>10} {'Time':>10}"
print(f"\n{header}")
print("-" * 72)

for name, res in all_results.items():
    dev = res.get("dev", {})
    dt = res.get("dev_test", {})
    acc = dev.get("accuracy", "—")
    ece = dev.get("ece", "—")
    dt_acc = dt.get("accuracy", "—") if dt else "—"
    elapsed = res.get("elapsed_min", "—")
    if isinstance(acc, float): acc = f"{acc:.4f}"
    if isinstance(ece, float): ece = f"{ece:.4f}"
    if isinstance(dt_acc, float): dt_acc = f"{dt_acc:.4f}"
    if isinstance(elapsed, (int, float)): elapsed = f"{elapsed:.1f}m"
    print(f"{name:<25} {acc:>10} {ece:>10} {dt_acc:>10} {elapsed:>10}")

print("-" * 72)

# ── Honest assessment ──
if "neural_baseline" in all_results and "nst_veri" in all_results:
    b_acc = all_results["neural_baseline"].get("dev", {}).get("accuracy")
    v_acc = all_results["nst_veri"].get("dev", {}).get("accuracy")
    if isinstance(b_acc, (int, float)) and isinstance(v_acc, (int, float)):
        delta = v_acc - b_acc
        print(f"\n  NST-VERI vs Baseline delta: {delta:+.4f}")
        if delta > 0.01:
            print("  → NST-VERI shows improvement over neural baseline")
        elif delta < -0.01:
            print("  → NST-VERI underperforms neural baseline (constraints may be too aggressive)")
        else:
            print("  → Results are within noise margin (~1%). No clear winner.")
    b_ece = all_results["neural_baseline"].get("dev", {}).get("ece")
    v_ece = all_results["nst_veri"].get("dev", {}).get("ece")
    if isinstance(b_ece, (int, float)) and isinstance(v_ece, (int, float)):
        ece_delta = v_ece - b_ece
        print(f"  Calibration delta (ECE): {ece_delta:+.4f}")
        if ece_delta < -0.005:
            print("  → NST-VERI is better calibrated (lower ECE is better)")
        elif ece_delta > 0.005:
            print("  → Baseline is better calibrated")
        else:
            print("  → Calibration is similar")

# ── Caveats ──
print(f"\n  CAVEATS:")
if DEVICE == "mps":
    print(f"  • Trained on Apple Silicon MPS (not A100)")
    print(f"  • Limited to {MAX_TRAIN or '???'} training samples")
    print(f"  • FP16 mixed precision (not bf16)")
elif DEVICE == "cpu":
    print(f"  • Trained on CPU — results are preliminary at best")
print(f"  • Gold evidence setting (IR retrieval not tested)")
print(f"  • Single seed — no variance estimate")
print(f"  • Max length 384 (some evidence may be truncated)")

# ── Save consolidated report ──
report = {
    "date": datetime.now().isoformat(),
    "device": DEVICE_NAME,
    "results": all_results,
}
with open("results_consolidated.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
print(f"\n  Consolidated report saved to results_consolidated.json")

  HONEST RESULTS COMPARISON
  Date: 2026-04-16 01:20
  Device: NVIDIA A100-SXM4-40GB (42GB)

Model                        Dev Acc    Dev ECE    DevTest       Time
------------------------------------------------------------------------
smoke                         0.3111     0.0626     0.2000          —
neural_baseline               0.8378     0.0401     0.8350          —
nst_veri                      0.8384     0.0423     0.8320          —
------------------------------------------------------------------------

  NST-VERI vs Baseline delta: +0.0006
  → Results are within noise margin (~1%). No clear winner.
  Calibration delta (ECE): +0.0021
  → Calibration is similar

  CAVEATS:
  • Gold evidence setting (IR retrieval not tested)
  • Single seed — no variance estimate
  • Max length 384 (some evidence may be truncated)

  Consolidated report saved to results_consolidated.json


In [9]:
# ============================================================
# Cell 7: Save & Commit Results
# ============================================================
import subprocess, os

# ── List output artifacts ──
artifacts = []
for f in os.listdir("."):
    if f.startswith("results_") and f.endswith(".json"):
        artifacts.append(f)
for d in os.listdir("."):
    if d.startswith("outputs_fever") and os.path.isdir(d):
        artifacts.append(d + "/")

print("Artifacts:")
for a in sorted(artifacts):
    print(f"  {a}")

# ── Optional: git commit results ──
try:
    status = subprocess.run(["git", "status", "--porcelain"],
                           capture_output=True, text=True, cwd=".")
    if status.stdout.strip():
        print(f"\n{len(status.stdout.strip().splitlines())} files changed")
        print("Run the following to commit results:")
        print('  git add -A && git commit -m "results: FEVER experiment run" && git push origin main')
    else:
        print("\nNo uncommitted changes.")
except Exception as e:
    print(f"Git check skipped: {e}")

print("\nDone. See results_consolidated.json for the full report.")

Artifacts:
  outputs_fever_neural_baseline/
  outputs_fever_nst_veri/
  results_baseline.json
  results_consolidated.json
  results_smoke.json
  results_veri.json

4 files changed
Run the following to commit results:
  git add -A && git commit -m "results: FEVER experiment run" && git push origin main

Done. See results_consolidated.json for the full report.
